# Layer 4: Maximum Scale — Up to 54 Qubits

Push the GME witness to the maximum available qubit count on IQM Emerald.

Also identifies the **bottleneck**: at which qubit count does certification fail, and why.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from src.backend import get_backend, get_best_grid, get_calibration_scores
from src.circuits.cluster_2d import build_cluster_2d_no_measure
from src.witnesses.gme_cluster import run_gme

import os
TOKEN = os.environ.get('IQM_TOKEN', None)
backend = get_backend(token=TOKEN, device='emerald')
print(f'Backend: {backend}')

In [ ]:
# Grid sizes to sweep — increase until GME fails
# Each job = 2 circuits × 10000 shots = within IQM limits
grid_sizes = [
    (2, 3),   #  6
    (3, 3),   #  9
    (3, 4),   # 12
    (4, 4),   # 16
    (4, 5),   # 20
    (5, 5),   # 25
    (5, 6),   # 30
    (6, 6),   # 36
    (6, 7),   # 42
    (6, 9),   # 54  ← full Emerald
]

SHOTS = 10000
scaling = []

for rows, cols in grid_sizes:
    n = rows * cols
    print(f'\n{rows}x{cols} = {n} qubits ...')
    try:
        if TOKEN:
            scores = get_calibration_scores(backend)
            layout = get_best_grid(backend, rows, cols, scores=scores)
        circ = build_cluster_2d_no_measure(rows, cols)
        res = run_gme(backend, circ, rows, cols, shots=SHOTS)
        scaling.append({
            'n': n, 'rows': rows, 'cols': cols,
            **res
        })
        status = 'GME CERTIFIED' if res['is_gme'] else 'not certified'
        print(f'  W = {res["W"]:.3f} / {res["W_ideal"]:.0f}  '
              f'bound={res["biseparable_bound"]}  '
              f'{res["significance_sigma"]:.1f}σ  {status}')
    except Exception as e:
        print(f'  ERROR: {e}')
        break

In [ ]:
# Main scaling figure
ns = [r['n'] for r in scaling]
W_vals = [r['W'] for r in scaling]
W_ideal = [r['W_ideal'] for r in scaling]
W_bound = [r['biseparable_bound'] for r in scaling]
sigmas = [r['significance_sigma'] for r in scaling]
gme_flags = [r['is_gme'] for r in scaling]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: W value
ax = axes[0]
ax.plot(ns, W_ideal, 'b--', label='Ideal W = n', linewidth=1.5)
ax.plot(ns, W_bound, 'r:', label='Biseparable bound = n-1', linewidth=1.5)
colors = ['#32a8a4' if g else '#e84545' for g in gme_flags]
ax.scatter(ns, W_vals, c=colors, s=80, zorder=5)
ax.plot(ns, W_vals, '-', color='gray', alpha=0.5)
ax.set_xlabel('Qubits (n)')
ax.set_ylabel('Witness value W')
ax.set_title('GME Witness W vs Qubit Count')
ax.legend()

# Panel 2: fraction of ideal
ax = axes[1]
fractions = [w/wi for w, wi in zip(W_vals, W_ideal)]
bound_fracs = [(wi-1)/wi for wi in W_ideal]
ax.plot(ns, bound_fracs, 'r:', label='GME threshold (bound/ideal)', linewidth=1.5)
ax.scatter(ns, fractions, c=colors, s=80, zorder=5)
ax.plot(ns, fractions, '-', color='gray', alpha=0.5)
ax.set_xlabel('Qubits (n)')
ax.set_ylabel('W / n  (fraction of ideal)')
ax.set_title('Normalised Witness Value')
ax.legend()
ax.set_ylim(0, 1.05)

plt.suptitle('GME Scaling on IQM Emerald — 2D Cluster States\n'
             'Green = GME certified, Red = below threshold',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('scaling_main.png', dpi=150, bbox_inches='tight')
plt.show()

max_gme = max((r['n'] for r in scaling if r['is_gme']), default=0)
print(f'\nMaximum GME-certified qubit count: {max_gme}')

In [ ]:
# Bottleneck analysis: where does certification fail?
print('=== Bottleneck Analysis ===')
print(f'{'n':>5} {'W/n':>8} {'σ':>8} {'GME':>8}')
print('-' * 35)
for r in scaling:
    frac = r['W'] / r['W_ideal']
    bound_frac = r['biseparable_bound'] / r['W_ideal']
    status = 'YES' if r['is_gme'] else 'NO'
    print(f'{r["n"]:>5} {frac:>8.4f} {r["significance_sigma"]:>8.2f} {status:>8}')

# Estimate at what qubit count fidelity drops below threshold
if len(scaling) >= 3:
    # Fit exponential decay to W/n
    from scipy.optimize import curve_fit
    def decay(n, a, b):
        return a * np.exp(-b * n)
    try:
        popt, _ = curve_fit(decay, ns, fractions, p0=[1.0, 0.01])
        n_threshold = -np.log(popt[0] - (max(bound_fracs) if bound_fracs else 0.99)) / popt[1]
        print(f'\nEstimated fidelity decay: f(n) ≈ {popt[0]:.3f} × exp(-{popt[1]:.4f} × n)')
        print(f'Primary bottleneck: gate fidelity decay with circuit depth/qubit count')
    except Exception:
        pass